# 04b - SARIMA Forecasting

**Objective**: Forecast using SARIMA (Seasonal ARIMA)

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load feature-engineered data
try:
    df = pd.read_csv('../data/clean/maize_features.csv', parse_dates=['date'])
except FileNotFoundError:
    df = pd.read_csv('data/clean/maize_features.csv', parse_dates=['date'])

# Use only date and price for SARIMA
ts_data = df[['date', 'price']].set_index('date')['price']

print(f'Time series length: {len(ts_data)}')
print(f'Date range: {ts_data.index.min()} to {ts_data.index.max()}')


Time series length: 180
Date range: 2006-01-01 00:00:00 to 2020-12-01 00:00:00


In [3]:
# Train-test split
train = df.iloc[:-12]
test = df.iloc[-12:]
print(f'Train: {len(train)}, Test: {len(test)}')

Train: 168, Test: 12


In [5]:
# Auto ARIMA to find best parameters
# Extract just the price values from train
train_prices = train['price'].values  # This gives you just the numerical values

stepwise_model = auto_arima(train_prices, start_p=1, start_q=1, max_p=3, max_q=3, m=12, start_P=0, seasonal=True, d=None, D=1, trace=True, error_action='ignore', suppress_warnings=True, stepwise=True)
print(f'\nBest model: {stepwise_model.order} x {stepwise_model.seasonal_order}')

Performing stepwise search to minimize aic
 ARIMA(1,0,1)(0,1,1)[12] intercept   : AIC=inf, Time=2.39 sec
 ARIMA(0,0,0)(0,1,0)[12] intercept   : AIC=1160.530, Time=0.07 sec
 ARIMA(1,0,0)(1,1,0)[12] intercept   : AIC=871.549, Time=0.93 sec
 ARIMA(0,0,1)(0,1,1)[12] intercept   : AIC=1018.629, Time=0.63 sec
 ARIMA(0,0,0)(0,1,0)[12]             : AIC=1167.604, Time=0.05 sec
 ARIMA(1,0,0)(0,1,0)[12] intercept   : AIC=907.941, Time=0.15 sec
 ARIMA(1,0,0)(2,1,0)[12] intercept   : AIC=851.387, Time=1.98 sec
 ARIMA(1,0,0)(2,1,1)[12] intercept   : AIC=inf, Time=9.13 sec
 ARIMA(1,0,0)(1,1,1)[12] intercept   : AIC=inf, Time=2.04 sec
 ARIMA(0,0,0)(2,1,0)[12] intercept   : AIC=1138.845, Time=1.58 sec
 ARIMA(2,0,0)(2,1,0)[12] intercept   : AIC=846.766, Time=2.13 sec
 ARIMA(2,0,0)(1,1,0)[12] intercept   : AIC=865.731, Time=0.62 sec
 ARIMA(2,0,0)(2,1,1)[12] intercept   : AIC=inf, Time=10.82 sec
 ARIMA(2,0,0)(1,1,1)[12] intercept   : AIC=inf, Time=2.16 sec
 ARIMA(3,0,0)(2,1,0)[12] intercept   : AIC=848.5

In [ ]:
# Train SARIMA with best parameters
# Use the same train_prices or extract again
model = SARIMAX(train_prices, order=stepwise_model.order, seasonal_order=stepwise_model.seasonal_order)
model_fit = model.fit(disp=False)
print('SARIMA model trained')
print(model_fit.summary())

SARIMA model trained
                                      SARIMAX Results                                      
Dep. Variable:                                   y   No. Observations:                  168
Model:             SARIMAX(2, 0, 2)x(2, 1, [], 12)   Log Likelihood                -415.940
Date:                             Thu, 12 Mar 2026   AIC                            845.881
Time:                                     18:59:44   BIC                            867.230
Sample:                                          0   HQIC                           854.552
                                             - 168                                         
Covariance Type:                               opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          1.7799      0.206      8.645      0.000       1.376       2.183
ar.L2 

In [8]:
# Forecast
forecast = model_fit.forecast(steps=12)
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Extract test prices
test_prices = test['price'].values

mae = mean_absolute_error(test_prices, forecast)
rmse = np.sqrt(mean_squared_error(test_prices, forecast))
mape = np.mean(np.abs((test_prices - forecast) / test_prices)) * 100
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')

MAE: 4.56
RMSE: 5.32
MAPE: 8.56%


In [11]:
from pathlib import Path
Path('../models/trained').mkdir(parents=True, exist_ok=True)

# Save model
import joblib
joblib.dump(model_fit, '../models/trained/sarima_maize_model.pkl')

# Fix: Get the actual dates from the original data
# Get the last 12 dates from the original data
last_dates = df['date'].iloc[-12:].values  # Use the original df with dates

forecast_df = pd.DataFrame({
    'date': last_dates,  # Use actual dates, not test.index
    'forecast': forecast  # forecast is already a numpy array
})
forecast_df.to_csv('../models/trained/sarima_forecast.csv', index=False)
print('Model saved')

Model saved
